# 🎙️ Proyecto Auditoria DIGI — Fase 1: Procesado de Audio

**Nodos cubiertos:** 1. ffmpeg → 2. DeepFilterNet 3 → 3. Silero VAD → 4. VoxLingua107 (LID)

**Cómo usar este notebook:**
1. Ejecuta la celda de instalación (Paso 1) — te pedirá reiniciar el runtime automáticamente
2. Tras el reinicio, ejecuta el resto de celdas en orden

---
> ⚠️ **Nota de versiones:** Fija las versiones exactas el día del despliegue en producción.

## ✅ Paso 0 — Verificar GPU

In [ ]:
import subprocess
resultado = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if resultado.returncode == 0:
    print('✅ GPU disponible:')
    for linea in resultado.stdout.split('\n'):
        if any(x in linea for x in ['Tesla','RTX','A100','T4','V100','L4']):
            print(' ', linea.strip())
            break
else:
    print('❌ Sin GPU — ve a: Entorno de ejecución → Cambiar tipo → GPU')

## 📦 Paso 1 — Instalar dependencias

**⚠️ IMPORTANTE:** Esta celda instala los paquetes y luego **reinicia el runtime automáticamente**. Es necesario para que DeepFilterNet cargue correctamente. Después del reinicio, continúa desde el Paso 2.

In [ ]:
import sys

# Verificar si ya se instaló (evitar doble reinicio)
try:
    from df.enhance import enhance
    print('✅ Dependencias ya instaladas — puedes continuar con el Paso 2')
except ImportError:
    print('📦 Instalando dependencias...')
    
    # Nodo 1: ffmpeg
    import subprocess
    subprocess.run(['ffmpeg', '-version'], capture_output=True)
    print('  ✓ ffmpeg disponible')
    
    # Nodo 2: DeepFilterNet 3
    !pip install -q deepfilternet
    print('  ✓ deepfilternet instalado')
    
    # Nodo 3: Silero VAD
    !pip install -q silero-vad
    print('  ✓ silero-vad instalado')
    
    # Nodo 4: SpeechBrain + VoxLingua107
    !pip install -q speechbrain
    print('  ✓ speechbrain instalado')
    
    # Audio utils
    !pip install -q torchaudio soundfile pydub
    print('  ✓ librerías de audio instaladas')
    
    print('\n🔄 Reiniciando runtime para activar las extensiones compiladas...')
    print('   Cuando reinicie, ejecuta las celdas desde el Paso 2 hacia abajo.')
    
    import os, time
    time.sleep(2)
    os.kill(os.getpid(), 9)  # Fuerza reinicio del kernel

## 🎵 Paso 2 — Subir archivo de audio

> Sube un archivo .ogg, .mp3 o .m4a de una llamada de prueba.

In [ ]:
from google.colab import files
import os

print('Selecciona tu archivo de audio...')
subidos = files.upload()

ARCHIVO_ORIGINAL = list(subidos.keys())[0]
print(f'\n✅ Archivo recibido: {ARCHIVO_ORIGINAL}')
print(f'   Tamaño: {os.path.getsize(ARCHIVO_ORIGINAL) / 1024:.1f} KB')

## 🔧 Nodo 1 — ffmpeg: Decodificar y normalizar

**¿Qué hace?** Convierte cualquier formato a WAV mono 16kHz con volumen normalizado. Es el formato estándar que necesitan todos los modelos de voz.

In [ ]:
import subprocess, os

ARCHIVO_WAV = 'audio_normalizado.wav'

cmd = ['ffmpeg', '-y', '-i', ARCHIVO_ORIGINAL,
       '-ac', '1', '-ar', '16000', '-af', 'loudnorm', ARCHIVO_WAV]

resultado = subprocess.run(cmd, capture_output=True, text=True)

if resultado.returncode == 0:
    print(f'✅ Nodo 1 completado')
    print(f'   {os.path.getsize(ARCHIVO_ORIGINAL)/1024:.1f} KB → {os.path.getsize(ARCHIVO_WAV)/1024:.1f} KB')
    print(f'   Formato: mono, 16kHz, volumen normalizado')
else:
    print(f'❌ Error en ffmpeg:\n{resultado.stderr[-500:]}')

## 🔇 Nodo 2 — DeepFilterNet 3: Reducción de ruido

**¿Qué hace?** Elimina ruido de fondo, estática y eco típicos de telefonía. Mejora mucho la precisión de Whisper.

In [ ]:
import torch
from df.enhance import enhance, init_df, load_audio, save_audio

ARCHIVO_LIMPIO = 'audio_sin_ruido.wav'

print('🔇 Cargando DeepFilterNet 3...')
model_df, df_state, _ = init_df()

print('   Procesando audio...')
audio, _ = load_audio(ARCHIVO_WAV, sr=df_state.sr())
audio_limpio = enhance(model_df, df_state, audio)
save_audio(ARCHIVO_LIMPIO, audio_limpio, df_state.sr())

del model_df
torch.cuda.empty_cache()

print(f'✅ Nodo 2 completado — ruido eliminado')
print(f'   Archivo: {os.path.getsize(ARCHIVO_LIMPIO)/1024:.1f} KB')

## ✂️ Nodo 3 — Silero VAD: Recortar silencios

**¿Qué hace?** Detecta cuándo hay voz y cuándo no. Elimina los silencios para que Whisper no pierda tiempo ni invente palabras donde no hay voz.

In [ ]:
import torch, torchaudio
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps, collect_chunks

ARCHIVO_VAD = 'audio_vad.wav'
SAMPLE_RATE = 16000

print('✂️ Cargando Silero VAD v5...')
model_vad = load_silero_vad()

wav = read_audio(ARCHIVO_LIMPIO, sampling_rate=SAMPLE_RATE)

timestamps = get_speech_timestamps(
    wav, model_vad, sampling_rate=SAMPLE_RATE,
    threshold=0.5,
    min_speech_duration_ms=250,
    min_silence_duration_ms=300,
)

audio_vad = collect_chunks(timestamps, wav)
torchaudio.save(ARCHIVO_VAD, audio_vad.unsqueeze(0), SAMPLE_RATE)

duracion_original = len(wav) / SAMPLE_RATE
duracion_vad = len(audio_vad) / SAMPLE_RATE
reduccion = (1 - duracion_vad / duracion_original) * 100

del model_vad

print(f'✅ Nodo 3 completado')
print(f'   Fragmentos de voz: {len(timestamps)}')
print(f'   {duracion_original:.1f}s → {duracion_vad:.1f}s (reducción de silencios: {reduccion:.1f}%)')

## 🌍 Nodo 4 — VoxLingua107: Identificar idioma

**¿Qué hace?** Detecta el idioma de la llamada. Si no es español, genera un flag de alerta en el reporte.

In [ ]:
import torchaudio, torch
from speechbrain.pretrained import EncoderClassifier

IDIOMA_ESPERADO = 'es'

print('🌍 Cargando VoxLingua107 (ECAPA)...')
classifier = EncoderClassifier.from_hparams(
    source='speechbrain/lang-id-voxlingua107-ecapa',
    savedir='tmp_lid'
)

signal, _ = torchaudio.load(ARCHIVO_VAD)
prediction = classifier.classify_batch(signal)
idioma_detectado = prediction[3][0].split(':')[0].strip()
confianza = float(prediction[1][0].max().exp()) * 100

flag_idioma = idioma_detectado != IDIOMA_ESPERADO

del classifier
torch.cuda.empty_cache()

print(f'\n✅ Nodo 4 completado')
print(f'   Idioma detectado: {idioma_detectado} ({confianza:.1f}% confianza)')
print(f'   {"⚠️ FLAG: idioma incorrecto para el territorio" if flag_idioma else "✅ Idioma correcto"}')

## 📋 Resumen — Fase 1 completada

In [ ]:
import json

resultado_fase1 = {
    'archivo_original': ARCHIVO_ORIGINAL,
    'archivo_salida': ARCHIVO_VAD,
    'duracion_original_s': round(duracion_original, 2),
    'duracion_procesada_s': round(duracion_vad, 2),
    'reduccion_silencios_pct': round(reduccion, 1),
    'idioma_detectado': idioma_detectado,
    'idioma_esperado': IDIOMA_ESPERADO,
    'confianza_idioma_pct': round(confianza, 1),
    'flags': {'idioma_incorrecto': flag_idioma}
}

print('=' * 55)
print('  FASE 1 — RESULTADO')
print('=' * 55)
for k, v in resultado_fase1.items():
    if k != 'flags':
        print(f'  {k}: {v}')
print()
print('  FLAGS:')
for flag, valor in resultado_fase1['flags'].items():
    print(f'    {flag}: {"⚠️ ACTIVO" if valor else "✅ OK"}')
print('=' * 55)
print(f'\n➡️  Listo para Fase 2: {ARCHIVO_VAD}')

with open('resultado_fase1.json', 'w') as f:
    json.dump(resultado_fase1, f, indent=2, ensure_ascii=False)
print('   Metadatos: resultado_fase1.json')